# ver6 환자 단위 5-Fold Fine-Tuning 비교

이 노트북은 `NonDemented vs VeryMildDemented` 환자 단위 조기 탐지 task에서 다음 두 학습 전략을 동일한 fold로 비교합니다.

- `full_finetune`: ver4 방식
- `partial_finetune`: ver5 방식

각 outer fold의 test 환자는 모델 학습, threshold 선택, early stopping에 사용하지 않습니다.

```text
전체 환자
  ├─ Outer train 환자
  │    ├─ Inner train: parameter 학습
  │    └─ Inner validation: best epoch 선택
  └─ Outer test 환자: 해당 fold 최종 평가
```

모든 성능 지표는 slice 단위가 아닌 환자 단위로 계산합니다.

## 1. 환경 설정

In [1]:
import os
import re
import gc
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
from contextlib import contextmanager

import numpy as np
import pandas as pd
import torch

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.executable}")
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"device: {device}")

Python: C:\Users\user\anaconda3\envs\alzheimer\python.exe
torch: 2.12.0.dev20260408+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070
device: cuda


## 2. 실험 Config

In [2]:
DATA_ROOT = Path(r"C:\Users\user\Desktop\alzheimer_dataset\Data")
OUTPUT_DIR = Path(r"C:\Users\user\alzheimer\patient_level_cv")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

NEGATIVE_CLASS = "NonDemented"
POSITIVE_CLASS = "VeryMildDemented"
TASK_NAME = "non_vs_verymild"

N_SPLITS = 5
INNER_VAL_RATIO = 0.15

# 전체 비교는 두 전략 × 5 folds = 10회 학습입니다.
EXPERIMENTS_TO_RUN = ["full_finetune", "partial_finetune"]
FOLDS_TO_RUN = [1, 2, 3, 4, 5]

# 실행 확인용으로 시간을 줄이려면 아래처럼 바꿀 수 있습니다.
# EXPERIMENTS_TO_RUN = ["full_finetune", "partial_finetune"]
# FOLDS_TO_RUN = [1]

TRAIN_AUG_MODE = "sepaug_4n"  # "original", "sepaug_3n", "sepaug_4n"
ROTATION_DEGREES = 10
SHIFT_TRANSLATE = (0.05, 0.05)
ZOOM_SCALE = (0.9, 1.1)
DETERMINISTIC_AUGMENTATION = True

BATCH_SIZE = 32
NUM_WORKERS = 0
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
USE_AMP = True
WEIGHT_DECAY = 1e-4
BALANCE_STRATEGY = "class_weight_sqrt"
EARLY_STOPPING_MIN_DELTA = 1e-4

# ver4와 ver5 설정을 그대로 비교합니다.
EXPERIMENT_CONFIGS = {
    "full_finetune": {
        "epochs": 12,
        "freeze_epochs": 3,
        "head_lr": 1e-3,
        "backbone_lr": 1e-5,
        "patience": 3,
        "unfreeze_mode": "all",
    },
    "partial_finetune": {
        "epochs": 8,
        "freeze_epochs": 3,
        "head_lr": 3e-4,
        "backbone_lr": 3e-6,
        "patience": 2,
        "unfreeze_mode": "last_stages",
    },
}

print(f"Task: {NEGATIVE_CLASS} vs {POSITIVE_CLASS}")
print(f"Outer folds: {N_SPLITS}")
print(f"Experiments: {EXPERIMENTS_TO_RUN}")
print(f"Folds to run: {FOLDS_TO_RUN}")

Task: NonDemented vs VeryMildDemented
Outer folds: 5
Experiments: ['full_finetune', 'partial_finetune']
Folds to run: [1, 2, 3, 4, 5]


## 3. 환자 Manifest 생성

In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PATIENT_PATTERN = re.compile(r"^(OAS1_\d+)", re.IGNORECASE)

rows = []
for target, class_name in enumerate([NEGATIVE_CLASS, POSITIVE_CLASS]):
    class_dir = DATA_ROOT / class_name
    assert class_dir.exists(), class_dir
    for image_path in sorted(class_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        patient_match = PATIENT_PATTERN.match(image_path.name)
        assert patient_match, image_path.name
        rows.append({
            "image_path": str(image_path),
            "class_name": class_name,
            "patient_id": patient_match.group(1).upper(),
            "target": target,
        })

manifest = pd.DataFrame(rows)
assert manifest.groupby("patient_id")["target"].nunique().max() == 1

patient_table = (
    manifest.groupby("patient_id", as_index=False)
    .agg(target=("target", "first"), class_name=("class_name", "first"), image_count=("image_path", "count"))
)

print(f"Images: {len(manifest):,}")
print(f"Patients: {len(patient_table):,}")
print(patient_table["target"].value_counts().sort_index())

Images: 80,947
Patients: 324
target
0    266
1     58
Name: count, dtype: int64


## 4. Dataset 및 환자 단위 평가 함수

In [4]:
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

weights = models.EfficientNet_B0_Weights.DEFAULT
preprocess = weights.transforms()

@contextmanager
def temporary_seed(seed):
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    try:
        random.seed(seed)
        np.random.seed(seed % (2**32 - 1))
        torch.manual_seed(seed)
        yield
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(torch_state)

class PatientSliceDataset(Dataset):
    def __init__(self, dataframe, preprocess, aug_mode="original", deterministic=True, seed=42):
        self.df = dataframe.reset_index(drop=True).copy()
        self.preprocess = preprocess
        self.deterministic = deterministic
        self.seed = seed

        if aug_mode == "original":
            self.output_types = ["original"]
        elif aug_mode == "sepaug_3n":
            self.output_types = ["rotation", "shift", "zoom"]
        elif aug_mode == "sepaug_4n":
            self.output_types = ["original", "rotation", "shift", "zoom"]
        else:
            raise ValueError(aug_mode)

        self.rotation = transforms.RandomRotation(ROTATION_DEGREES)
        self.shift = transforms.RandomAffine(degrees=0, translate=SHIFT_TRANSLATE)
        self.zoom = transforms.RandomAffine(degrees=0, scale=ZOOM_SCALE)

    def __len__(self):
        return len(self.df) * len(self.output_types)

    def _augment(self, image, aug_type):
        if aug_type == "original":
            return image
        if aug_type == "rotation":
            return self.rotation(image)
        if aug_type == "shift":
            return self.shift(image)
        if aug_type == "zoom":
            return self.zoom(image)
        raise ValueError(aug_type)

    def __getitem__(self, idx):
        multiplier = len(self.output_types)
        base_idx = idx // multiplier
        aug_idx = idx % multiplier
        row = self.df.iloc[base_idx]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.deterministic:
            with temporary_seed(self.seed + base_idx * 1009 + aug_idx * 9176):
                image = self._augment(image, self.output_types[aug_idx])
        else:
            image = self._augment(image, self.output_types[aug_idx])

        return (
            self.preprocess(image),
            torch.tensor(int(row["target"]), dtype=torch.long),
            row["patient_id"],
        )

def build_model():
    model = models.efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    return model.to(device)

def aggregate_patient_predictions(patient_ids, labels, probabilities, threshold=0.5):
    grouped_probs = defaultdict(list)
    grouped_labels = {}
    for patient_id, label, prob in zip(patient_ids, labels, probabilities):
        grouped_probs[patient_id].append(prob)
        grouped_labels[patient_id] = label
    ordered_ids = sorted(grouped_probs)
    patient_probs = np.array([np.mean(grouped_probs[pid]) for pid in ordered_ids])
    patient_labels = np.array([grouped_labels[pid] for pid in ordered_ids])
    patient_preds = (patient_probs >= threshold).astype(int)
    return ordered_ids, patient_labels, patient_probs, patient_preds

def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    specificity = cm[0, 0] / max(cm[0].sum(), 1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auroc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
    }

## 5. Fine-Tuning 비교 함수

In [5]:
from sklearn.model_selection import StratifiedKFold, train_test_split

def configure_trainable_layers(model, mode):
    for parameter in model.parameters():
        parameter.requires_grad = False

    if mode == "head_only":
        pass
    elif mode == "all":
        for parameter in model.features.parameters():
            parameter.requires_grad = True
    elif mode == "last_stages":
        for parameter in model.features[7].parameters():
            parameter.requires_grad = True
        for parameter in model.features[8].parameters():
            parameter.requires_grad = True
    else:
        raise ValueError(mode)

    for parameter in model.classifier.parameters():
        parameter.requires_grad = True

def make_optimizer(model, config):
    head_ids = {id(p) for p in model.classifier.parameters()}
    head_params, backbone_params = [], []
    for parameter in model.parameters():
        if not parameter.requires_grad:
            continue
        if id(parameter) in head_ids:
            head_params.append(parameter)
        else:
            backbone_params.append(parameter)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": config["backbone_lr"]})
    groups.append({"params": head_params, "lr": config["head_lr"]})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def make_class_weight(inner_train_patients):
    counts_series = inner_train_patients["target"].value_counts().sort_index()
    counts = np.array([counts_series.get(0, 0), counts_series.get(1, 0)], dtype=np.float64)
    if BALANCE_STRATEGY == "class_weight_sqrt":
        values = 1.0 / np.sqrt(np.maximum(counts, 1))
        values /= values.mean()
        return torch.tensor(values, dtype=torch.float32, device=device)
    return None

def evaluate_model(model, loader, threshold=0.5):
    model.eval()
    patient_ids, labels_all, probabilities_all = [], [], []
    with torch.inference_mode():
        for images, labels, batch_patient_ids in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=(USE_AMP and torch.cuda.is_available())):
                logits = model(images)
            probabilities = logits.softmax(dim=1)[:, 1].cpu().numpy()
            patient_ids.extend(list(batch_patient_ids))
            labels_all.extend(labels.numpy().tolist())
            probabilities_all.extend(probabilities.tolist())

    ids, y_true, y_prob, y_pred = aggregate_patient_predictions(
        patient_ids, labels_all, probabilities_all, threshold=threshold
    )
    return calculate_metrics(y_true, y_prob, threshold), ids, y_true, y_prob

def train_one_fold(experiment_name, fold_number, outer_train_patients, outer_test_patients):
    config = EXPERIMENT_CONFIGS[experiment_name]
    run_seed = SEED + fold_number * 100
    seed_everything(run_seed)

    inner_train_patients, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    inner_train_ids = set(inner_train_patients["patient_id"])
    inner_val_ids = set(inner_val_patients["patient_id"])
    outer_test_ids = set(outer_test_patients["patient_id"])
    assert inner_train_ids.isdisjoint(inner_val_ids)
    assert inner_train_ids.isdisjoint(outer_test_ids)
    assert inner_val_ids.isdisjoint(outer_test_ids)

    train_df = manifest[manifest["patient_id"].isin(inner_train_ids)].copy()
    val_df = manifest[manifest["patient_id"].isin(inner_val_ids)].copy()
    test_df = manifest[manifest["patient_id"].isin(outer_test_ids)].copy()

    train_dataset = PatientSliceDataset(
        train_df, preprocess, TRAIN_AUG_MODE, DETERMINISTIC_AUGMENTATION, run_seed
    )
    val_dataset = PatientSliceDataset(val_df, preprocess, "original", True, run_seed)
    test_dataset = PatientSliceDataset(test_df, preprocess, "original", True, run_seed)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )

    model = build_model()
    configure_trainable_layers(model, "head_only")
    optimizer = make_optimizer(model, config)
    class_weight = make_class_weight(inner_train_patients)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and torch.cuda.is_available()))

    best_val_auroc = -1.0
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    fine_tuning_started = False

    print(
        f"\n[{experiment_name} fold {fold_number}] "
        f"inner train={len(inner_train_patients)}, val={len(inner_val_patients)}, "
        f"outer test={len(outer_test_patients)}"
    )

    for epoch in range(1, config["epochs"] + 1):
        if epoch == config["freeze_epochs"] + 1:
            configure_trainable_layers(model, config["unfreeze_mode"])
            optimizer = make_optimizer(model, config)
            fine_tuning_started = True
            epochs_without_improvement = 0

        model.train()
        loss_sum, total = 0.0, 0
        for images, labels, patient_ids in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(USE_AMP and torch.cuda.is_available())):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * labels.size(0)
            total += labels.size(0)

        val_metrics, _, _, _ = evaluate_model(model, val_loader)
        improved = val_metrics["auroc"] > best_val_auroc + EARLY_STOPPING_MIN_DELTA
        if improved:
            best_val_auroc = val_metrics["auroc"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        elif fine_tuning_started:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:02d}/{config['epochs']} | loss {loss_sum / max(total, 1):.4f} | "
            f"val AUROC {val_metrics['auroc']:.4f} AUPRC {val_metrics['auprc']:.4f} | "
            f"early {epochs_without_improvement}/{config['patience']}"
        )

        if fine_tuning_started and epochs_without_improvement >= config["patience"]:
            print("Early stopping")
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    model = model.to(device)

    test_metrics, test_ids, y_true, y_prob = evaluate_model(model, test_loader)
    checkpoint_path = CHECKPOINT_DIR / f"{TASK_NAME}_{experiment_name}_fold{fold_number}.pt"
    torch.save({
        "model_state_dict": best_state,
        "experiment": experiment_name,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "best_val_auroc": best_val_auroc,
        "test_metrics": test_metrics,
    }, checkpoint_path)

    result = {
        "experiment": experiment_name,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "val_auroc": best_val_auroc,
        "test_patients": len(test_ids),
        **test_metrics,
    }
    print("Test:", result)

    del model, optimizer, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 6. 동일한 5 Folds로 두 전략 실행

In [ ]:
outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
patient_indices = np.arange(len(patient_table))
patient_targets = patient_table["target"].to_numpy()

fold_assignments = []
all_results = []

for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
    outer_cv.split(patient_indices, patient_targets), start=1
):
    outer_train_patients = patient_table.iloc[outer_train_idx].reset_index(drop=True)
    outer_test_patients = patient_table.iloc[outer_test_idx].reset_index(drop=True)

    for patient_id in outer_test_patients["patient_id"]:
        fold_assignments.append({"patient_id": patient_id, "outer_test_fold": fold_number})

    if fold_number not in FOLDS_TO_RUN:
        continue

    print(
        f"\n===== OUTER FOLD {fold_number} =====\n"
        f"train patients={len(outer_train_patients)}, test patients={len(outer_test_patients)}\n"
        f"test class counts={outer_test_patients['target'].value_counts().sort_index().to_dict()}"
    )

    for experiment_name in EXPERIMENTS_TO_RUN:
        result = train_one_fold(
            experiment_name,
            fold_number,
            outer_train_patients,
            outer_test_patients,
        )
        all_results.append(result)

        pd.DataFrame(all_results).to_csv(
            OUTPUT_DIR / "finetuning_comparison_partial_results.csv",
            index=False,
            encoding="utf-8-sig",
        )

fold_assignment_df = pd.DataFrame(fold_assignments)
fold_assignment_df.to_csv(
    OUTPUT_DIR / f"{TASK_NAME}_outer_5fold_assignments_seed{SEED}.csv",
    index=False,
    encoding="utf-8-sig",
)

results_df = pd.DataFrame(all_results)
print(results_df)


===== OUTER FOLD 1 =====
train patients=259, test patients=65
test class counts={0: 54, 1: 11}

[full_finetune fold 1] inner train=220, val=39, outer test=65
Epoch 01/12 | loss 0.4756 | val AUROC 0.9643 AUPRC 0.9238 | early 0/3
Epoch 02/12 | loss 0.4695 | val AUROC 0.8884 AUPRC 0.6049 | early 0/3
Epoch 03/12 | loss 0.4692 | val AUROC 0.8571 AUPRC 0.5956 | early 0/3
Epoch 04/12 | loss 0.2864 | val AUROC 0.9777 AUPRC 0.9087 | early 0/3
Epoch 05/12 | loss 0.1206 | val AUROC 1.0000 AUPRC 1.0000 | early 0/3
Epoch 06/12 | loss 0.0577 | val AUROC 0.9911 AUPRC 0.9683 | early 1/3
Epoch 07/12 | loss 0.0318 | val AUROC 1.0000 AUPRC 1.0000 | early 2/3
Epoch 08/12 | loss 0.0191 | val AUROC 0.9955 AUPRC 0.9821 | early 3/3
Early stopping
Test: {'experiment': 'full_finetune', 'fold': 1, 'best_epoch': 5, 'val_auroc': 1.0, 'test_patients': 65, 'accuracy': 0.8615384615384616, 'precision': 0.6, 'sensitivity': 0.5454545454545454, 'specificity': np.float64(0.9259259259259259), 'f1': 0.5714285714285714, 'ma

## 7. 평균 및 표준편차 비교

In [ ]:
assert not results_df.empty, "실행된 fold 결과가 없습니다."

metric_columns = [
    "accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1",
    "macro_f1",
    "auroc",
    "auprc",
]

summary_rows = []
for experiment_name, group in results_df.groupby("experiment"):
    row = {"experiment": experiment_name, "folds": len(group)}
    for metric in metric_columns:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std(ddof=1)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "finetuning_comparison_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\n[Fold별 결과]")
display(results_df.sort_values(["experiment", "fold"]))
print("\n[평균 ± 표준편차]")
display(summary_df)
print(f"summary saved: {summary_path}")

for _, row in summary_df.iterrows():
    print(
        f"{row['experiment']}: "
        f"AUROC {row['auroc_mean']:.4f} ± {row['auroc_std']:.4f} | "
        f"AUPRC {row['auprc_mean']:.4f} ± {row['auprc_std']:.4f} | "
        f"Sensitivity {row['sensitivity_mean']:.4f} ± {row['sensitivity_std']:.4f} | "
        f"Specificity {row['specificity_mean']:.4f} ± {row['specificity_std']:.4f}"
    )

## 8. 해석 기준

Fine-tuning 방식을 결정할 때 단일 fold 최고 점수가 아니라 다음 기준을 함께 봅니다.

1. 평균 AUROC와 평균 AUPRC
2. 조기 탐지에 중요한 평균 sensitivity
3. fold 간 표준편차
4. specificity의 과도한 하락 여부
5. best epoch가 일관되게 너무 늦지 않는지

두 전략의 평균이 비슷하면 parameter가 적고 과적합 위험이 낮은 `partial_finetune`을 선택합니다. `full_finetune`이 여러 지표에서 일관되게 높고 표준편차도 크지 않을 때만 전체 fine-tuning을 선택합니다.